In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

In [3]:
from script.hypothesis_tests import load_last3_df, load_covid_df, load_capacity_df

last3_df = load_last3_df().copy()
covid_df = load_covid_df().copy()
capacity_df = load_capacity_df().copy()

print("last3_df:", last3_df.shape)
print("covid_df:", covid_df.shape)
print("capacity_df:", capacity_df.shape)



last3_df: (3578, 29)
covid_df: (3578, 25)
capacity_df: (1343, 22)


In [4]:
def interpret_pvalue(p_value, alpha=0.05):
    if p_value < alpha:
        return f"p-value = {p_value:.6f} < {alpha} -> H0 can be denied."
    else:
        return f"p-value = {p_value:.6f} >= {alpha} -> H0 can't be denied."

def cramers_v(contingency_table):
    chi2, _, _, _ = stats.chi2_contingency(contingency_table)
    n = contingency_table.to_numpy().sum()
    r, k = contingency_table.shape
    return np.sqrt(chi2 / (n * min(r - 1, k - 1)))


In [5]:
# Hypothesis Test 1 (General) - Chi Square
# H0: There isn't any correlation between home form and result of the match.
# H1: There is a correlation between home form and result of the match.

contingency_form = (last3_df.groupby(["home_form", "result_match"]).size().unstack(fill_value=0))

chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_form)
cramers = cramers_v(contingency_form)

print("Contingency Table:")
display(contingency_form)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"Cramer's V: {cramers:.4f}")
print(interpret_pvalue(p_value))

Contingency Table:


result_match,Away Win,Draw,Home Win
home_form,,,
bad,662,425,590
good,134,148,307
medium,416,329,567


Chi-square statistic: 72.1389
Degrees of freedom: 4
Cramer's V: 0.1004
p-value = 0.000000 < 0.05 -> H0 can be denied.


In [6]:
# Hypothesis Test 2 (last 3 match performances) - Pearson
# H0: There isn't any correlation between point difference and home win rates.
# H1: As point difference increases, home win rate increases.

r_stat, p_value = stats.pearsonr(last3_df["point_diff"], last3_df["home_win"])

summary_df = (last3_df.groupby("home_win")["point_diff"].agg(["count", "mean", "median"]).reset_index())

summary_df["home_win"] = summary_df["home_win"].map({0: "No", 1: "Yes"})

display(summary_df)

print(f"Pearson correlation coefficient (r): {r_stat:.4f}")
print(interpret_pvalue(p_value))

,home_win,count,mean,median
0,No,2114,-0.658940,0.0
1,Yes,1464,0.614071,0.0


Pearson correlation coefficient (r): 0.1906
p-value = 0.000000 < 0.05 -> H0 can be denied.


In [7]:
# Hypothesis Test 3 (Covid vs. Normal season) - Z test
# H0: Home win rates are equal in the COVID and normal seasons.
# H1: Home win rates are different in the COVID and normal seasons.

home_win_counts = covid_df.groupby("season_type")["home_win"].sum().reindex(["covid", "normal"])
match_counts = covid_df.groupby("season_type")["home_win"].count().reindex(["covid", "normal"])
home_win_rates = home_win_counts / match_counts

covid_summary = pd.DataFrame({"home_wins": home_win_counts,"matches": match_counts,"home_win_rate": home_win_rates})

z_stat, p_value = proportions_ztest(count=home_win_counts.values,nobs=match_counts.values)

display(covid_summary)

print(f"Z statistic: {z_stat:.4f}")
print(interpret_pvalue(p_value))

,home_wins,matches,home_win_rate
season_type,,,
covid,728,1826,0.398686
normal,736,1752,0.420091


Z statistic: -1.3018
p-value = 0.192986 >= 0.05 -> H0 can't be denied.


In [8]:
# Hypothesis Test 4 (Stadium Capacity) - Chi Square
# H0: There isn't any correlation between capacity interval and match result.
# H1: There is a correlation between capacity interval and match result.

contingency_capacity = (
    capacity_df
    .groupby(["capacity_interval", "result"], observed=False)
    .size()
    .unstack(fill_value=0))

chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_capacity)

capacity_summary = (
    capacity_df
    .groupby("capacity_interval",observed=False)["home_win"]
    .agg(["count", "mean"])
    .reset_index())

capacity_summary.columns = ["capacity_interval", "matches", "home_win_rate"]

print("Contingency Table:")
display(contingency_capacity)
display(capacity_summary)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(interpret_pvalue(p_value))

Contingency Table:


result,A,D,H
capacity_interval,,,
0-20K,90,67,90
20K-40K,239,157,207
40K+,111,115,267


,capacity_interval,matches,home_win_rate
0,0-20K,247,0.364372
1,20K-40K,603,0.343284
2,40K+,493,0.541582


Chi-square statistic: 54.3511
Degrees of freedom: 4
p-value = 0.000000 < 0.05 -> H0 can be denied.


In [8]:
## General Conclusion of Hypothesis Tests
#In this hypothesis testing section, the main findings from the EDA were tested statistically.
#The results showed that home team form and recent performance are significantly related to match outcome. In particular, as the recent point differential increases, the probability of a home win also increases. This supports the patterns observed in the EDA.
#The comparison between COVID and normal seasons showed that the home win rates were different numerically, but this difference was not statistically significant at the 0.05 level. Therefore, the EDA pattern in this part was not strongly supported by hypothesis testing.
#Finally, stadium capacity and match result were found to be significantly related. This suggests that larger stadium environments may be associated with stronger home advantage.
#Overall, hypothesis testing confirmed most of the important EDA findings, while also showing that not every visual difference is statistically significant.
